In [0]:
def load_bronze_incremental(source_files,table_name,checkpoint_suffix):
    df=(
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("cloudFiles.schemaLocation",f"/Volumes/retailworks/bronze/source_files/_schema_{checkpoint_suffix}")
        .option("header",True)
        .load(f"/Volumes/retailworks/bronze/source_files/{source_files}")
    )
    (
        df.writeStream.format("delta")
        .option("checkpointLocation",f"/Volumes/retailworks/bronze/source_files/_checkpoint_{checkpoint_suffix}")
        .outputMode("append")
        .trigger(availableNow=True)
        .table(f"retailworks.bronze.{table_name}")
    )
    
    
load_bronze_incremental("crm_customers.csv","crm_customers_raw","crm")
load_bronze_incremental("erp_products.csv","erp_products_raw","erp")
load_bronze_incremental("orders.csv","orders_raw","orders")


In [0]:
def load_bronze_incremental(source_file, table_name, checkpoint_suffix):
    processed_marker = f"/Volumes/retailworks/bronze/source_files/_processed_{checkpoint_suffix}.txt"
    
    try:
        already_loaded = dbutils.fs.head(processed_marker).splitlines()
    except Exception:
        already_loaded = []
    
    if source_file in already_loaded:
        print(f"{source_file} already loaded, skipping")
        return
    
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"/Volumes/retailworks/bronze/source_files/{source_file}")
    )
    
    df.write.format("delta").mode("append").saveAsTable(f"retailworks.bronze.{table_name}")
    
    already_loaded.append(source_file)
    dbutils.fs.put(processed_marker, "\n".join(already_loaded), overwrite=True)
    print(f"Loaded {source_file} into retailworks.bronze.{table_name}")


load_bronze_incremental("crm_customers.csv", "crm_customers_raw", "crm")
load_bronze_incremental("erp_products.csv", "erp_products_raw", "erp")
load_bronze_incremental("orders.csv", "orders_raw", "orders")

let me break down exactly what this code is doing and why each piece exists:

**The core problem it solves:** if you just did `spark.read.csv(...)` and wrote it every time, running the notebook twice would **duplicate every row** — since batch reads don't know what's "new" vs. "already loaded." Incremental loading means: *only load files you haven't processed yet.*

**How each part achieves that:**

```python
processed_marker = f"/Volumes/retailworks/bronze/source_files/_processed_{checkpoint_suffix}.txt"
```
This creates a small text file that acts as a **memory/log** — it records which source files have already been loaded. One marker file per source (`_crm`, `_erp`, `_orders`) so they're tracked independently.

```python
try:
    already_loaded = dbutils.fs.head(processed_marker).splitlines()
except Exception:
    already_loaded = []
```
This tries to **read that memory** — "what have I already loaded before?" If the marker file doesn't exist yet (first run ever), it fails and we just start with an empty list.

```python
if source_file in already_loaded:
    print(f"{source_file} already loaded, skipping")
    return
```
This is the actual **incremental check** — "have I seen this exact file before? If yes, stop here, don't reload it."

```python
df.write.format("delta").mode("append").saveAsTable(...)
```
Only reaches this line if the file is genuinely new — reads it and **appends** (not overwrites) it to the Bronze Delta table, adding to what's already there rather than replacing it.

```python
already_loaded.append(source_file)
dbutils.fs.put(processed_marker, "\n".join(already_loaded), overwrite=True)
```
After successfully loading, it **updates the memory** — writes the filename back into the marker file, so next time this file is correctly recognized as "already done."

**Why this matters conceptually (ties back to what you learned):** this is literally the same problem Autoloader's `checkpointLocation` solves — Autoloader tracks "which files have I already processed" internally and automatically. This code does the exact same job manually, just visibly, since Autoloader's Structured Streaming path is currently broken on Free Edition. It's a fair, honest way to demonstrate you understand *why* incremental loading works, even without the managed tool doing it for you.